In [9]:

from datetime import datetime, timedelta
import pandas as pd
import numpy as np


def ImportOMNI_Month(StartDateTime, EndDateTime, Resolution = '1min', Columns = 'All'):

    """
    Finds local OMNI Data files, if not available attempts to download from https://cdaweb.gsfc.nasa.gov/pub/data/omni/high_res_omni/
    Rules of the Road: https://omniweb.sci.gsfc.nasa.gov/html/citing.html

    Arguments:

    StartDateTime - datetime for the start of the requested data interval
    EndDateTime -   datetime for the end of the requested data interval
    Resolution -    1min or 5min -> only 1min implemented

    Returns:

    Data -  DataFrame

    """

    import urllib.request
    #from datetime import datetime, timedelta

    Year = datetime.strftime(StartDateTime, '%Y')
    DOY = datetime.strftime(StartDateTime, '%j')
    Month = datetime.strftime(StartDateTime, '%m')
    DayOfMonth = datetime.strftime(StartDateTime, '%d')

    Header = ['Year', 'Day', 'Hour', 'Minute', 'B_IMF_ScID', 'Plasma_ScID', 'IMFAvPoints', 'PlasmaAvPoints',
        'PercentInterp', 'Timeshift', 'RMSTimeshift', 'RMSPhaseFrontNormal', 'TimeBetweenObs',
        'B', 'B_X_GSM', 'B_Y_GSE', 'B_Z_GSE', 'B_Y_GSM', 'B_Z_GSM', 'RMSBScalar', 'RMSFieldVector',
        'V', 'V_X_GSE', 'V_Y_GSE', 'V_Z_GSE', 'n_p', 'T', 'P', 'E', 'Beta', 'AlfvenMach', 'X_SC_GSE',
        'Y_SC_GSE', 'Z_SC_GSE', 'X_BSN_GSE', 'Y_BSN_GSE', 'Z_BSN_GSE', 'AE', 'AL', 'AU', 'SymD', 'SymH',
        'AsyD', 'AsyH', 'PCN', 'MagnetosonicMach', '10MeVProton', '30MeVProton', '60MeVProton']

    try:
        import pickle
        FNameLocalDir = r"/Users/kylerichardaffleck/Downloads/"
        FNameLocal = FNameLocalDir + 'OMNI_1min_' +Year+Month+'.p'
        Data = pickle.load(open(FNameLocal, 'rb'))

        Data = Data[(Data.DateTime >= StartDateTime) & (Data.DateTime <= EndDateTime)]
 
    except IOError:
        try:
            FNameLocalDir =  r"/Users/kylerichardaffleck/Downloads/"
            FNameLocal = FNameLocalDir + 'OMNI_1min_' +Year+Month+'.asc'
            Data = pd.read_csv(FNameLocal, sep = '\s+', names = Header, header = None)
            Data['DateTime'] = Data.apply(lambda row: datetime(int(row.Year), 1, 1)+timedelta(days = int(row.Day) - 1)+ timedelta(seconds = row.Hour*60*60+row.Minute*60), axis = 1)
            # print('Found Local Data')

        except IOError:

            print('Local data not found -> downloading from  https://cdaweb.gsfc.nasa.gov/pub/data/omni/high_res_omni/')
            FName = 'https://cdaweb.gsfc.nasa.gov/pub/data/omni/high_res_omni/monthly_1min/omni_min'+Year+Month+'.asc'
            urllib.request.urlretrieve(FName, FNameLocal)
            print('Downloaded')
            Data = pd.read_csv(FNameLocal, sep = '\s+', names = Header, header = None)
            Data['DateTime'] = Data.apply(lambda row: datetime(int(row.Year), 1, 1)+timedelta(days = int(row.Day) - 1)+ timedelta(seconds = row.Hour*60*60+row.Minute*60), axis = 1)

        Data = Data.replace(99.99, np.nan)
        Data = Data.replace(999.9, np.nan)
        Data = Data.replace(999.99, np.nan)
        Data = Data.replace(9999.99, np.nan)
        Data = Data.replace(99999.9, np.nan)
        Data = Data.replace(9999999., np.nan)

        Data.index = Data['DateTime']

        import pickle

        FNameLocalDir = r"/Users/kylerichardaffleck/Downloads/"
        FNameLocal = FNameLocalDir + 'OMNI_1min_' +Year+Month+'.p'
        with open(FNameLocal, "wb") as output_file:
            pickle.dump(Data, output_file)

        Data = Data[(Data.DateTime >= StartDateTime) & (Data.DateTime <= EndDateTime)]

    # print(Data.index.values[0])
    # print(Data.columns)
    if Columns != 'All':

        Data = Data[Columns]

    # print(Data.columns)

    # import sys; sys.exit()

    return Data

def ImportOMNI(StartDateTime, EndDateTime, Resolution = '1min', Columns = 'All'):

    #### Always specify columns if importing large amount of data, will significantly speed up (20 years full data is ~4Gb)

    from tqdm import tqdm

    DataMonths = ReturnYearMonths(StartDateTime, EndDateTime)

    if len(DataMonths) == 1:
        # print(StartDateTime, EndDateTime)
        Data = ImportOMNI_Month(StartDateTime, EndDateTime, Resolution = '1min', Columns = Columns)
    else:
        print('Importing Multiple Months')
        for n, MonthDT in enumerate(tqdm(DataMonths)):
            if n == 0:
                Data_Temp = ImportOMNI_Month(StartDateTime, DataMonths[1]-timedelta(seconds = 1.), Resolution = '1min', Columns = Columns)
                Data = Data_Temp
            elif MonthDT.month == EndDateTime.month:
                Data_Temp = ImportOMNI_Month(MonthDT, EndDateTime, Resolution = '1min', Columns = Columns)
                Data = pd.concat([Data, Data_Temp], sort = False)
            else:
                Data_Temp = ImportOMNI_Month(MonthDT, DataMonths[n+1]-timedelta(seconds = 1.), Resolution = '1min', Columns = Columns)
                Data = pd.concat([Data, Data_Temp], sort = False)     

    return Data 


def ReturnYearMonths(StartDateTime, EndDateTime):

    # StartDateTime.month
    # EndDateTime.month

    from dateutil.rrule import rrule, MONTHLY

    # ListOfMonths = np.arange(StartDateTime.month, EndDateTime.month+1)

    ListOfYearMonths = [dt for dt in rrule(MONTHLY, dtstart = StartDateTime, until = EndDateTime)]

    return ListOfYearMonths

<>:55: SyntaxWarning: invalid escape sequence '\s'
<>:65: SyntaxWarning: invalid escape sequence '\s'
<>:55: SyntaxWarning: invalid escape sequence '\s'
<>:65: SyntaxWarning: invalid escape sequence '\s'
/var/folders/xk/35xssqr514g4r9k0tm7wppxw0000gn/T/ipykernel_70968/729958836.py:55: SyntaxWarning: invalid escape sequence '\s'
  Data = pd.read_csv(FNameLocal, sep = '\s+', names = Header, header = None)
/var/folders/xk/35xssqr514g4r9k0tm7wppxw0000gn/T/ipykernel_70968/729958836.py:65: SyntaxWarning: invalid escape sequence '\s'
  Data = pd.read_csv(FNameLocal, sep = '\s+', names = Header, header = None)


In [29]:
# Define start and end dates
start_date = datetime(2024, 11, 1, 16, 00)
end_date = datetime(2024, 11, 1, 18, 00)

# Import data using ImportOMNI_Month
solar_wind_values = ImportOMNI(start_date, end_date, Resolution='5min', Columns='All')
solar_wind_values["V"]

DateTime
2024-11-01 16:00:00    420.0
2024-11-01 16:01:00    420.0
2024-11-01 16:02:00    420.0
2024-11-01 16:03:00      NaN
2024-11-01 16:04:00    417.4
                       ...  
2024-11-01 17:56:00    412.4
2024-11-01 17:57:00      NaN
2024-11-01 17:58:00      NaN
2024-11-01 17:59:00      NaN
2024-11-01 18:00:00    416.2
Name: V, Length: 121, dtype: float64

In [23]:
print(solar_wind_values.dtypes)

Year                            int64
Day                             int64
Hour                            int64
Minute                          int64
B_IMF_ScID                      int64
Plasma_ScID                     int64
IMFAvPoints                     int64
PlasmaAvPoints                  int64
PercentInterp                   int64
Timeshift                       int64
RMSTimeshift                    int64
RMSPhaseFrontNormal           float64
TimeBetweenObs                  int64
B                             float64
B_X_GSM                       float64
B_Y_GSE                       float64
B_Z_GSE                       float64
B_Y_GSM                       float64
B_Z_GSM                       float64
RMSBScalar                    float64
RMSFieldVector                float64
V                             float64
V_X_GSE                       float64
V_Y_GSE                       float64
V_Z_GSE                       float64
n_p                           float64
T           